# 16.1 팀 프로젝트: 발표 — 캡스톤 프로토콜 실습 노트북

[![Open In Colab: 팀 프로젝트 프로토콜](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter16_1_capstone_protocol.ipynb)

책 16.1절의 **프로젝트 절차**를 Pendulum-v1 하나에 압축해서 한 흐름으로 통과시킵니다:
1. **(M1) 문제 정의** — MDP 5요소와 리턴의 이론적 상한/하한, **베이스라인(무작위 정책) 측정**
2. **(M2) 구현** — Chapter 11.2의 PPO(Actor-Critic + GAE + 클리핑)를 Pendulum에 적용
3. **(M3) 시드 ≥3 프로토콜** — 같은 코드를 시드 0/1/2로 재실행, 학습 도중 리턴 비교
4. **학습 후 평가** — 학습 도중 리턴이 아닌, **탐색 없이(결정적) 고정된 정책**을 재는 평가 프로토콜

핵심 질문: *"학습 곡선이 올라갔다"는 것과 "배운 정책이 실제로 무작위보다 나은가"는 다른 문장인가?* — 이 노트북의 마지막 결과가 정직하게 답합니다.

In [1]:
import os, random, math, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

# 한국어 라벨 폰트 (없으면 기본 폰트로 넘어가도 출력은 됨)
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
plt.rcParams["font.family"] = kr[0] if kr else "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"
print("그림 저장:", IMG)
torch.set_num_threads(4)

그림 저장: /home/smhan/book-ml/kor/src/images


## 1. (M1) 문제 정의: Pendulum-v1의 MDP와 리턴의 이론적 범위

진자를 수직(위쪽)으로 오래 유지하는 연속 제어 과제입니다. 관찰은 3차원 `[cosθ, sinθ, θ̇]`, 행동은 1차원 토크 `a ∈ [-2, +2]` Nm, 에피소드는 200스텝(시간 10초)입니다.

보상은 매 스텝 **항상 음수**입니다:

$$r = -\theta^2 - 0.1\,\dot{\theta}^2 - 0.001\,a^2$$

여기서 θ는 수직에서 잰 각도(라디안), θ̇는 각속도입니다. 세 항 모두 "수직이고 멈춰 있는 상태(θ=0, θ̇=0)"에서 0이 되므로:
- **이론적 상한 = 0** (진자를 영원히 수직으로 고정 — 실제로는 잡음 때문에 도달 불가)
- 매 스텝 하한: θ² ≤ π² ≈ 9.87, θ̇²는 제어 에너지로 제한됨 → **에피소드당 리턴은 대략 −2000 안팎에서 −100 사이**를 떠돌며, 0에 가까울수록 좋음

In [2]:
env = gym.make("Pendulum-v1")
u = env.unwrapped
print("관찰 공간: ", env.observation_space.shape, "(cosθ, sinθ, θ̇)")
print("행동 공간: ", env.action_space.shape, " 토크 범위 [", env.action_space.low, env.action_space.high, "]")
print("에피소드 길이: ", env.spec.max_episode_steps, "스텝 (10초)")
print("보상: r = -θ² - 0.1θ̇² - 0.001a² (항상 음수, 0에 가까울수록 좋음)")

관찰 공간:  (3,) (cosθ, sinθ, θ̇)
행동 공간:  (1,)  토크 범위 [ [-2.] [2.] ]
에피소드 길이:  200 스텝 (10초)
보상: r = -θ² - 0.1θ̇² - 0.001a² (항상 음수, 0에 가까울수록 좋음)


### 1.1 베이스라인: 무작위 정책의 평균 리턴

8.1절 M2에서 요구했던 것 그대로 — **"배운 정책이 무작위보다 나은가"를 판정할 제로점**을 먼저 측정합니다. 무작위 정책(매 스텝 균등 무작위 토크)으로 100에피소드:

In [3]:
import numpy as _np
_np.random.seed(0)  # 베이스라인도 재현 가능하게 시드 고정
baseline = []
for ep in range(100):
    state, _ = env.reset(seed=1000 + ep)
    total = 0.0
    while True:
        state, r, term, trunc, _ = env.step(env.action_space.sample())
        total += r
        if term or trunc:
            break
    baseline.append(total)
baseline_mean, baseline_sd = float(np.mean(baseline)), float(np.std(baseline))
print(f"무작위 정책 (100 에피소드): mean = {baseline_mean:.1f}  sd = {baseline_sd:.1f}")
print(f"범위: [{np.min(baseline):.1f}, {np.max(baseline):.1f}]")
print("→ 이 숫자가 '제로점'이다. PPO의 최종 평가가 이걸 못 넘으면 '배웠다'고 말하지 못한다.")

무작위 정책 (100 에피소드): mean = -1274.1  sd = 300.0
범위: [-1844.3, -791.9]
→ 이 숫자가 '제로점'이다. PPO의 최종 평가가 이걸 못 넘으면 '배웠다'고 말하지 못한다.


## 2. (M2) 구현: Chapter 11.2의 PPO를 그대로

강의 코드(11.2절 `ActorCritic` + GAE)를 재사용합니다 — 8.1절 FAQ "코드를 처음부터 써야 하나요?"의 답과 같은 이유로, 프로젝트는 강의 코드의 재사용을 전제로 합니다. `mu` 출력이 **결정적 평가용 평균**이기도 하다는 점에 주목하세요(§4).

In [4]:
class ActorCritic(nn.Module):
    def __init__(self, state_dim, act_dim, act_high):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(state_dim, 64), nn.Tanh(),
                                   nn.Linear(64, 64), nn.Tanh())
        self.mu = nn.Linear(64, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)  # exp() 후 약 0.61
        self.v = nn.Linear(64, 1)
        self.act_high = act_high
    def forward(self, x):
        h = self.trunk(x)
        mu = torch.tanh(self.mu(h)) * self.act_high
        std = torch.exp(self.log_std)
        return mu, std, self.v(h).squeeze(-1)

def compute_gae(rewards, values, last_value, gamma=0.99, lam=0.95):
    T = len(rewards); adv = [0.0] * T; last_gae = 0.0
    for t in reversed(range(T)):
        next_v = values[t + 1] if t < T - 1 else last_value
        delta = rewards[t] + gamma * next_v - values[t]
        last_gae = delta + gamma * lam * last_gae
        adv[t] = last_gae
    return adv, [a + v for a, v in zip(adv, values)]

def train_ppo(seed, n_iter=300, steps=400):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env = gym.make("Pendulum-v1")
    ac = ActorCritic(3, 1, env.action_space.high[0])
    opt = torch.optim.Adam(ac.parameters(), lr=3e-4)
    GAMMA, LAM, EPS, N_EPOCH, MB, ENT_COEF, VAL_COEF = 0.99, 0.95, 0.2, 4, 64, 0.01, 0.5
    rets, cursor = [], 0
    t0 = time.time()
    for it in range(n_iter):
        state, _ = env.reset(seed=seed + it)
        states, actions, logps, rewards, values = [], [], [], [], []
        for _ in range(steps):
            st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                mu, std, v = ac(st)
            dist = torch.distributions.Normal(mu, std)
            a = dist.sample()
            states.append(state); actions.append(a); logps.append(dist.log_prob(a).sum())
            next_state, r, term, trunc, _ = env.step(a.squeeze(0).numpy())
            rewards.append(r); values.append(v.item())
            if term or trunc:  # Pendulum: 200스텝마다 truncation
                rets.append(float(np.sum(rewards[cursor:])))
                cursor = len(rewards); state, _ = env.reset()
            else:
                state = next_state
        states_t = torch.tensor(np.asarray(states), dtype=torch.float32)
        actions_t = torch.tensor(np.asarray(actions), dtype=torch.float32)
        logps_t = torch.stack(logps)
        rewards_t = torch.tensor(rewards, dtype=torch.float32)
        values_t = torch.tensor(values, dtype=torch.float32)
        with torch.no_grad():
            last_v = ac(states_t[-1:])[2].item()
        adv, rets_gae = compute_gae(rewards_t.tolist(), values_t.tolist(), last_v, GAMMA, LAM)
        adv_t = torch.tensor(adv, dtype=torch.float32)
        ret_t = torch.tensor(rets_gae, dtype=torch.float32)
        adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)
        idx = torch.randperm(len(rewards))
        for _ in range(N_EPOCH):  # 같은 데이터를 4에폭 재사용 후 버림 (11.2절)
            for start in range(0, len(rewards), MB):
                b = idx[start:start + MB]
                mu, std, v = ac(states_t[b])
                dist = torch.distributions.Normal(mu, std)
                new_logps = dist.log_prob(actions_t[b]).sum(-1)
                ratio = torch.exp(new_logps - logps_t[b])
                pol = -torch.min(ratio * adv_t[b], torch.clamp(ratio, 1 - EPS, 1 + EPS) * adv_t[b]).mean()
                loss = pol + VAL_COEF * F.mse_loss(v, ret_t[b]) - ENT_COEF * dist.entropy().sum(-1).mean()
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
                opt.step()
    return ac, rets, time.time() - t0

print("train_ppo 준비 완료 (11.2절 코드 재사용, seed 파라미터로 시드 제어)")

train_ppo 준비 완료 (11.2절 코드 재사용, seed 파라미터로 시드 제어)


## 3. (M3) 시드 ≥3 프로토콜: 같은 코드를 세 번

8.1절 §"비교 프로토콜"의 연속제어 버전: **같은 코드·같은 하이퍼파라미터(γ=0.99, λ=0.95, ε_clip=0.2, lr=3e-4)를 시드 0/1/2로 전부 다시 학습**합니다. 각 시드 300반복 × 400스텝 = 12만 스텝(노트북 CPU 기준 시드당 약 25초). 시드는 "실험을 다시 하는" 것이 아니라 "우연인지 재검사하는" 것입니다.

In [5]:
results = {}
for seed in [0, 1, 2]:
    ac, rets, dt = train_ppo(seed)
    results[seed] = {"ac": ac, "rets": rets, "time": dt,
                     "first10": float(np.mean(rets[:10])),
                     "last10": float(np.mean(rets[-10:]))}
    print(f"시드 {seed}: 에피소드 {len(rets)}, 처음10={results[seed]['first10']:.1f}, "
          f"마지막10={results[seed]['last10']:.1f}  (학습 {dt:.0f}초)")

시드 0: 에피소드 600, 처음10=-777.2, 마지막10=-673.5  (학습 22초)


시드 1: 에피소드 600, 처음10=-865.9, 마지막10=-732.3  (학습 22초)


시드 2: 에피소드 600, 처음10=-914.5, 마지막10=-668.3  (학습 22초)


## 4. 학습 후 평가: "곡선이 올라갔다"가 아니라 "정책이 실제로 나은가"

여기서 8.1절의 핵심 원리가 **연속제어 버전**으로 등장합니다. 8.1절(표 기반)의 평가는 "학습 중 쓰던 \(\varepsilon\)-greedy에서 ε을 0으로" 바꿨지만, 연속 정책(정규분포)에는 ε이 없습니다 — 대응하는 원리는 **행동을 샘플링(\(\mu + \sigma z\))하지 않고 평균 \(\mu\)를 그대로 내보내는 것(결정적 실행)**입니다.

같은 정책 세 개(시드 0/1/2)를 세 가지 방식으로 재 봅니다:
- **학습 도중**: 학습 중 마지막 10에피소드(탐색 노이즈 \(\sigma\) 포함, PPO가 실제로 행동하던 방식)
- **결정적 평가**: \(a = \mu(s)\) (탐색 0, 50에피소드) — 8.1절의 "ε=0 greedy 평가"에 해당
- **베이스라인**: §1.1의 무작위 정책

In [6]:
def evaluate(ac, mode="det", n_ep=50, seed=0):
    """mode: 'det' (a=μ, 결정적) | 'sample' (a~N(μ,σ), 학습 중과 같은 방식)"""
    env = gym.make("Pendulum-v1")
    rets = []
    for ep in range(n_ep):
        state, _ = env.reset(seed=5000 + seed * 100 + ep)
        total = 0.0
        while True:
            st = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                mu, std, _ = ac(st)
            a = mu.numpy().squeeze(0) if mode == "det" else (mu + std * torch.randn_like(mu)).numpy().squeeze(0)
            state, r, term, trunc, _ = env.step(a)
            total += r
            if term or trunc:
                break
        rets.append(total)
    return float(np.mean(rets)), float(np.std(rets))

eval_table = {}
for seed, d in results.items():
    det, det_sd = evaluate(d["ac"], "det", seed=seed)
    samp, samp_sd = evaluate(d["ac"], "sample", seed=seed)
    eval_table[seed] = {"det": det, "det_sd": det_sd, "samp": samp, "samp_sd": samp_sd}
    print(f"시드 {seed}:  학습중 마지막10={d['last10']:8.1f}   결정적 평가={det:8.1f}(sd {det_sd:.0f})   "
          f"샘플 평가={samp:8.1f}(sd {samp_sd:.0f})")

print(f"\n베이스라인(무작위): {baseline_mean:.1f} (sd {baseline_sd:.0f})")
print("\n읽기: '학습 도중 마지막10'과 '결정적 평가'는 같은 정책의 두 얼굴이다 —")
print("      학습 도중 숫자는 탐색 노이즈와 12만 스텝 예산의 한계를 함께 담고 있다.")

시드 0:  학습중 마지막10=  -673.5   결정적 평가= -1357.9(sd 210)   샘플 평가= -1287.6(sd 181)


시드 1:  학습중 마지막10=  -732.3   결정적 평가= -1454.2(sd 186)   샘플 평가= -1399.4(sd 162)


시드 2:  학습중 마지막10=  -668.3   결정적 평가= -1398.2(sd 239)   샘플 평가= -1367.0(sd 167)

베이스라인(무작위): -1274.1 (sd 300)

읽기: '학습 도중 마지막10'과 '결정적 평가'는 같은 정책의 두 얼굴이다 —
      학습 도중 숫자는 탐색 노이즈와 12만 스텝 예산의 한계를 함께 담고 있다.


## 5. 발표 자료용 한 장: 학습 곡선 + 평가 프로토콜 비교

좌: 시드 0/1/2의 학습 곡선(연회색=에피소드별, 실선=25에피소드 이동평균). 시드마다 모양이 다르면 정상(§3). 우: **같은 세 정책**을 세 가지 프로토콜로 잰 결과(시드 평균 ± 표준편차). "학습 곡선이 올라갔다"는 문장의 진짜 의미는 오른쪽 패널에서 판단한다.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
colors = {0: "#4a90d9", 1: "#d95f4a", 2: "#4aa66a"}
ax = axes[0]
for seed, d in results.items():
    rets = np.array(d["rets"])
    ep = np.arange(1, len(rets) + 1)
    sm = np.convolve(rets, np.ones(25) / 25, mode="valid")
    ax.plot(ep, rets, color=colors[seed], alpha=0.12, lw=0.8)
    ax.plot(ep[24:], sm, color=colors[seed], lw=1.8, label=f"시드 {seed} (이동평균)")
ax.axhline(baseline_mean, color="black", ls="--", lw=1.2, label=f"무작위 베이스라인 {baseline_mean:.0f}")
ax.set_xlabel("에피소드"); ax.set_ylabel("에피소드 리턴 (0에 가까울수록 좋음)")
ax.set_title("학습 곡선 (시드 0/1/2, 12만 스텝)"); ax.legend(fontsize=9)

ax = axes[1]
metrics = ["무작위 베이스라인", "학습 도중 마지막10", "결정적 평가 (a=μ)", "샘플 평가 (a~N(μ,σ))"]
means = [baseline_mean,
         float(np.mean([d["last10"] for d in results.values()])),
         float(np.mean([e["det"] for e in eval_table.values()])),
         float(np.mean([e["samp"] for e in eval_table.values()]))]
sds = [baseline_sd, float(np.std([d["last10"] for d in results.values()])),
       float(np.std([e["det"] for e in eval_table.values()])),
       float(np.std([e["samp"] for e in eval_table.values()]))]
bars = ax.barh(range(len(metrics)), means, xerr=sds, height=0.55,
               color=["#888", colors[0], "#2c3e50", "#7f8c8d"], ecolor="black", capsize=5)
ax.set_yticks(range(len(metrics)), metrics)
ax.invert_yaxis()
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("평균 에피소드 리턴 (0에 가까울수록 좋음)")
ax.set_title("같은 정책, 다른 측정 방식 (시드 평균 ± sd)")

plt.tight_layout()
plt.savefig(f"{IMG}/ch16_1_ppo_seed_curves.svg", bbox_inches="tight")
plt.show()

## 6. 정리: 이 결과가 보고서에 어떻게 쓰이는가

오른쪽 패널을 그대로 "수식적 정당화 + 정직한 결과" 절로 번역하면 됩니다:

- **학습 도중 리턴**(마지막 10)이 무작위 베이스라인보다 높은 것은 맞지만, 그것이 "배운 정책"의 성능이 아니다 — PPO는 학습 내내 \(\sigma\) 노이즈가 섞인 행동을 했기 때문이다(11.2절의 "에피소드 리턴 자체의 분산은 줄일 수 없다").
- **결정적 평가**(a=μ)는 12만 스텝 예산으로는 여전히 베이스라인 근처(또는 이하)에 있을 수 있다. 이것은 실패가 아니라 **학습 예산의 한계**의 정직한 측정 — 13~14절에서 로봇 시뮬레이션에 수백만 스텝을 배정하는 이유가 바로 이것이다.
- 8.1절의 "학습 도중/학습 후 분리" 원칙이 연속제어에서는 **"샘플 정책의 리턴 / 결정적 정책의 리턴" 분리**로 번역됨을 보고서에 명시하는 것 — 이것이 이 프로젝트의 방법론적 산출물이다.